### Tutorial 7  Transfer Learning Task: Feature Extraction (ResNet50) and Fine-tuning (VGG16)
In this section,compare ResNet50 feature extraction with VGG16 and implement fine-tuning strategies.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
import copy

# Standard Preprocessing for ImageNet models
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#### 1. ResNet50 Feature Extraction
Freeze all weights and only train the final classification layer.

In [ ]:
# Use modern Weights enum instead of deprecated pretrained parameter
resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in resnet50.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
num_ftrs = resnet50.fc.in_features
resnet50.fc = nn.Linear(num_ftrs, 2)
resnet50 = resnet50.to(device)

print("ResNet50 loaded for feature extraction with modern weights.")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 144MB/s]


ResNet50 loaded for feature extraction with modern weights.


#### 2. VGG16 Fine-tuning with Optimization Hints
Unfreeze the last convolutional block to allow the model to adapt to specific medical image features.

In [ ]:
# Use modern Weights enum for VGG16
vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

# Unfreeze more layers: Specifically the last block of VGG16
for param in vgg16.features[:24]:
    param.requires_grad = False
for param in vgg16.features[24:]:
    param.requires_grad = True

# Modify classifier
vgg16.classifier[6] = nn.Linear(4096, 2)
vgg16 = vgg16.to(device)

# Optimization strategies for fine-tuning
optimizer_vgg = optim.Adam(vgg16.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

print("VGG16 loaded for fine-tuning with modern weights.")

VGG16 loaded for fine-tuning with modern weights.


#### 3. Early Stopping Logic
This helper function prevents overfitting by stopping training when validation loss stops improving.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

#### 4. Training and Evaluation Loop
This function integrates the early stopping logic and performs training and validation for comparison.

In [ ]:
def train_and_compare(model, model_name, criterion, optimizer, train_loader, val_loader, num_epochs=10):
    early_stopping = EarlyStopping(patience=3, min_delta=0.01)
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)
            print(f'{model_name} {phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val':
                early_stopping(epoch_loss)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())

        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

    model.load_state_dict(best_model_wts)
    return model, best_acc

print("Training function defined. You can now pass your DataLoaders to this function to compare models.")

Training function defined. You can now pass your DataLoaders to this function to compare models.


#### 4. Model Comparison and Training
Now prepare a training loop to compare the Feature Extraction performance of ResNet50 against the Fine-tuning of VGG16 and also include placeholders for a custom model comparison.

In [ ]:
def train_model(model, criterion, optimizer, early_stopping, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        # Placeholder for training logic using your DataLoader
        # running_loss = 0.0
        # ...

        # Validation phase
        model.eval()
        val_loss = 0.0 # Placeholder for actual val_loss calculation

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch}")
            break
    return model

# Initialize Early Stopping
es = EarlyStopping(patience=3)

print("Ready to begin training and comparison.")

Ready to begin training and comparison.
